# 004 · Running Processes with subprocess

> **Standard Library › System & CLI** · Python 3.14

`subprocess` starts other programs, feeds them input, captures their output and checks their exit codes — the job of `Process.Start` and `ProcessStartInfo` in .NET. One high-level function, `subprocess.run`, covers almost every case; `Popen` is there when you need to stream.

**In this notebook you will learn:**

- `subprocess.run` with `capture_output`, `text`, `check` and `timeout`
- exit codes, `CalledProcessError` and `TimeoutExpired`
- why you pass arguments as a **list** and avoid `shell=True` (command injection)
- sending input with `input=`, and controlling `env=` and `cwd=`
- streaming output line by line with `Popen`, and `shlex.quote` / `shlex.split`

## .NET mental model

| .NET | Python |
|---|---|
| `new ProcessStartInfo(file) { ArgumentList = {...} }` | `[program, arg1, arg2]` list |
| `Process.Start(psi)` + `WaitForExit()` | `subprocess.run(args)` |
| `RedirectStandardOutput = true` + `ReadToEnd()` | `capture_output=True` → `result.stdout` |
| `RedirectStandardInput = true` + write | `input="..."` |
| `process.ExitCode` | `result.returncode` |
| throw if `ExitCode != 0` (manual) | `check=True` → `CalledProcessError` |
| `WaitForExit(ms)` + `Kill()` | `timeout=` → `TimeoutExpired` (child is killed) |
| `psi.Environment["X"] = "v"` | `env={...}` |
| `psi.WorkingDirectory` | `cwd=...` |
| `UseShellExecute = true` / `cmd /c` | `shell=True` (avoid) |
| `process.OutputDataReceived` events | `Popen` + iterate over `proc.stdout` |
| `await process.WaitForExitAsync()` | `asyncio.create_subprocess_exec` |

To stay portable and offline, every example uses **the Python interpreter itself** as the child program: `[sys.executable, "-c", "some code"]`. `sys.executable` is the absolute path of the Python running this notebook *(see Standard Library › System & CLI › 001 · The sys Module)*.

## `subprocess.run` — the 90% function

`run` starts the process, waits for it to finish and returns a `CompletedProcess`. By default the child's output goes straight to *our* stdout/stderr; `capture_output=True` collects it instead, and `text=True` decodes bytes to `str`.

```csharp
// C#
var psi = new ProcessStartInfo("python") { RedirectStandardOutput = true };
psi.ArgumentList.Add("-c");
psi.ArgumentList.Add("print('hello from the child')");
using var process = Process.Start(psi)!;
string output = process.StandardOutput.ReadToEnd();
process.WaitForExit();
Console.WriteLine($"{process.ExitCode}: {output}");
```

In [ ]:
import os
import subprocess
import sys

result = subprocess.run([sys.executable, "-c", "print('hello from the child')"],
                        capture_output=True, text=True)

print(type(result).__name__)
print(result.returncode)          # 0 — success
print(repr(result.stdout))        # 'hello from the child\n'
print(repr(result.stderr))        # ''

Without `text=True` you get `bytes`, which is right for binary output (images, archives). For text, prefer setting `encoding="utf-8"` explicitly (it implies text mode) — `text=True` alone uses the locale's encoding, which differs on Windows. For a child Python, `PYTHONIOENCODING=utf-8` makes it *write* UTF-8 too.

In [ ]:
raw = subprocess.run([sys.executable, "-c", "print('héllo')"], capture_output=True)
print(type(raw.stdout).__name__, raw.stdout)                  # bytes

decoded = subprocess.run([sys.executable, "-c", "print('héllo')"],
                         capture_output=True, encoding="utf-8",
                         env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(type(decoded.stdout).__name__, decoded.stdout.strip())  # str

## Exit codes and `check=True`

A non-zero exit code is **not** an exception by default — just like .NET, where you must inspect `ExitCode`. Pass `check=True` to raise `CalledProcessError` on failure; the exception carries the return code and any captured output.

In [ ]:
failing = [sys.executable, "-c", "import sys; print('partial output'); sys.exit('disk full')"]

result = subprocess.run(failing, capture_output=True, text=True)
print(result.returncode, repr(result.stderr))      # 1 'disk full\n' — no exception

try:
    subprocess.run(failing, capture_output=True, text=True, check=True)
except subprocess.CalledProcessError as e:
    # str(e) includes the full command — here that's the interpreter's absolute path, so we print parts
    print(f"{type(e).__name__}: returned non-zero exit status {e.returncode}")
    print("stdout:", e.stdout.strip(), "| stderr:", e.stderr.strip(), "| cmd[-1]:", e.cmd[-1][:20] + "…")

`result.check_returncode()` does the same check after the fact. And `subprocess.check_output(args)` is a shortcut for "run, check, and return stdout":

In [ ]:
version = subprocess.check_output([sys.executable, "-c", "import sys; print(sys.version_info[0])"],
                                  text=True)
print(version.strip())    # 3

⚠️ **Gotcha:** a missing program is a different failure: there is no exit code because the process never started. You get `FileNotFoundError` (C#: `Win32Exception`):

In [ ]:
try:
    subprocess.run(["definitely-not-a-real-program-xyz"])
except FileNotFoundError as e:
    print(f"{type(e).__name__}: {e.strerror}")

## Timeouts

`timeout=` (seconds) stops waiting, **kills the child** and raises `TimeoutExpired`. Without it, a hung child hangs your program forever.

```csharp
// C#
if (!process.WaitForExit(500)) process.Kill(entireProcessTree: true);
```

In [ ]:
import time

start = time.perf_counter()
try:
    subprocess.run([sys.executable, "-c", "import time; time.sleep(10)"], timeout=0.5)
except subprocess.TimeoutExpired as e:
    print(f"{type(e).__name__}: timed out after {e.timeout}s")
print(f"waited ≈ {time.perf_counter() - start:.1f}s, not 10s")

## Arguments: pass a list, not a string

With a list, each element reaches the child as **one argument**, exactly as written — no quoting, no shell, no surprises. It is the equivalent of `ProcessStartInfo.ArgumentList` (rather than the single `Arguments` string).

In [ ]:
show_args = [sys.executable, "-c", "import sys; print(sys.argv[1:])"]

result = subprocess.run(show_args + ["file with spaces.txt", "--name=O'Brien", "$HOME", "*.txt"],
                        capture_output=True, text=True)
print(result.stdout.strip())      # every argument arrives verbatim — nothing expanded

### ⚠️ `shell=True` and command injection

`shell=True` hands one string to the system shell (`/bin/sh` or `cmd.exe`), which then interprets spaces, quotes, `;`, `|`, `$VAR`, wildcards… If any part of that string comes from a user, a file or a network request, they can run **arbitrary commands**. Here a harmless `echo` shows the problem (POSIX shells only):

In [ ]:
user_input = "report.txt; echo INJECTED: this ran as a second command"

if os.name == "posix":
    unsafe = subprocess.run(f"echo processing {user_input}", shell=True,
                            capture_output=True, text=True)
    print(unsafe.stdout)          # two commands ran!

safe = subprocess.run(show_args + [user_input], capture_output=True, text=True)
print(safe.stdout.strip())        # one harmless argument

Rules of thumb:

- Use a **list** and leave `shell=False` (the default).
- Only use `shell=True` for a fixed string you wrote yourself and that needs shell features (pipes, globbing) — and usually Python can do those parts better (`pathlib.Path.glob`, chaining with `input=`).
- If you truly must build a shell command from data, quote every piece with `shlex.quote` (POSIX shells only).

## `shlex.quote` and `shlex.split`

`shlex` understands POSIX shell syntax:

- `shlex.split(cmd)` turns a command-line string into an argument list (handy for config files that store a command as text);
- `shlex.quote(s)` escapes one value for safe inclusion in a shell string;
- `shlex.join(list)` is the inverse of `split` — great for logging the command you ran.

In [ ]:
import shlex

print(shlex.split('convert "my photo.png" --quality 90 -o out.jpg'))
print(shlex.quote(user_input))                       # wrapped in single quotes
print(shlex.join(["grep", "-r", "two words", "src/"]))

if os.name == "posix":
    quoted = subprocess.run(f"echo processing {shlex.quote(user_input)}", shell=True,
                            capture_output=True, text=True)
    print(quoted.stdout.strip())                     # one echo, no second command

⚠️ **Gotcha:** `shlex` follows POSIX rules; `cmd.exe` and PowerShell quote differently. That is one more reason to pass a list, which works the same everywhere.

## Sending input with `input=`

`input=` writes a string (with `text=True`) or bytes to the child's stdin and closes it — like setting `RedirectStandardInput` and writing to `process.StandardInput`.

In [ ]:
sort_lines = "import sys; print(''.join(sorted(sys.stdin)), end='')"

result = subprocess.run([sys.executable, "-c", sort_lines],
                        input="cherry\napple\nbanana\n", capture_output=True, text=True)
print(result.stdout)

Chaining processes like a shell pipe (`a | b`) is just: run the first, pass its `stdout` as the second's `input`.

In [ ]:
produce = subprocess.run([sys.executable, "-c", "for n in (3, 1, 2): print(n)"],
                         capture_output=True, text=True, check=True)
consume = subprocess.run([sys.executable, "-c", sort_lines],
                         input=produce.stdout, capture_output=True, text=True, check=True)
print(consume.stdout.split())     # ['1', '2', '3']

## Environment and working directory: `env=` and `cwd=`

- `env=` **replaces** the child's whole environment. To add or override a variable, copy `os.environ` first *(see Standard Library › System & CLI › 002 · Environment Variables and Platform Info)*.
- `cwd=` sets the child's working directory (C#: `WorkingDirectory`) without changing ours.

In [ ]:
import tempfile
from pathlib import Path

child_code = "import os; from pathlib import Path; print(os.environ.get('APP_MODE'), Path.cwd().name)"

with tempfile.TemporaryDirectory() as tmp:
    work_dir = Path(tmp) / "job-42"
    work_dir.mkdir()
    result = subprocess.run([sys.executable, "-c", child_code],
                            env={**os.environ, "APP_MODE": "batch"}, cwd=work_dir,
                            capture_output=True, text=True, check=True)
    print(result.stdout.strip())       # batch job-42

print(os.environ.get("APP_MODE"))      # None — our own environment is unchanged

## Redirect options: `DEVNULL` and merging stderr

Instead of `capture_output=True` you can set `stdout=` / `stderr=` individually: `subprocess.PIPE` (capture), `subprocess.DEVNULL` (discard), an open file, or for stderr `subprocess.STDOUT` (merge into stdout, like `2>&1`).

In [ ]:
noisy = "import sys; print('out'); print('err', file=sys.stderr)"

merged = subprocess.run([sys.executable, "-c", noisy],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(merged.stdout.split())            # both streams in one: ['out', 'err'] (order may vary)

quiet = subprocess.run([sys.executable, "-c", noisy],
                       stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
print(repr(quiet.stdout), quiet.stderr.strip())   # None err

## Streaming output with `Popen`

`run` waits until the child has finished. For long-running children where you want each line **as it arrives** (progress output, logs), use `Popen` directly and iterate over its `stdout` — the counterpart of handling `OutputDataReceived`.

```csharp
// C#
process.OutputDataReceived += (_, e) => { if (e.Data is not null) Console.WriteLine($"got: {e.Data}"); };
process.Start();
process.BeginOutputReadLine();
await process.WaitForExitAsync();
```

In [ ]:
worker = """
import time
for step in range(1, 4):
    print(f"step {step}/3 done", flush=True)   # flush, or the pipe buffers everything
    time.sleep(0.1)
"""

start = time.perf_counter()
with subprocess.Popen([sys.executable, "-c", worker],
                      stdout=subprocess.PIPE, text=True) as proc:
    for line in proc.stdout:                        # yields lines as they are produced
        print(f"[{time.perf_counter() - start:.1f}s] got: {line.rstrip()}")
print("exit code:", proc.returncode)

⚠️ **Gotcha:** when its stdout is a pipe, a child Python (like most programs) **block-buffers** its output, so you would get everything at the end. The child must flush (`flush=True`, `python -u`, or `PYTHONUNBUFFERED=1`).

⚠️ **Gotcha:** if you capture *both* stdout and stderr with `Popen`, do not read one to the end and then the other — the child can block on a full stderr pipe while you wait on stdout (deadlock). Use `proc.communicate()` (which `run` uses internally), or merge with `stderr=subprocess.STDOUT`.

In [ ]:
with subprocess.Popen([sys.executable, "-c", sort_lines],
                      stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                      text=True) as proc:
    out, err = proc.communicate(input="b\na\n", timeout=5)
print(out.split(), repr(err), proc.returncode)

**Tip:** in async code, use `asyncio.create_subprocess_exec(...)` and `await proc.communicate()` — the equivalent of `WaitForExitAsync` — so the event loop is not blocked *(see Standard Library › Concurrency › 005 · asyncio in Depth)*.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Run and wait | `Process.Start(psi)` + `WaitForExit()` | `subprocess.run([prog, *args])` |
| Capture output | `RedirectStandardOutput` + `ReadToEnd()` | `capture_output=True, text=True` → `.stdout` |
| Exit code | `process.ExitCode` | `result.returncode` |
| Fail on non-zero | manual check | `check=True` → `CalledProcessError` |
| Just get stdout | — | `subprocess.check_output(args, text=True)` |
| Timeout | `WaitForExit(ms)` + `Kill()` | `timeout=` → `TimeoutExpired` |
| Arguments | `ArgumentList.Add(...)` | a list of strings |
| Shell command | `UseShellExecute` / `cmd /c` | `shell=True` — avoid with untrusted data |
| Quote for a shell | — | `shlex.quote(s)` |
| Split a command string | — | `shlex.split(cmd)` |
| Write to stdin | `RedirectStandardInput` + write | `input="..."` |
| Environment | `psi.Environment[...]` | `env={**os.environ, ...}` |
| Working directory | `psi.WorkingDirectory` | `cwd=...` |
| Discard output | redirect and ignore | `stdout=subprocess.DEVNULL` |
| Stream lines | `OutputDataReceived` | `Popen(..., stdout=PIPE)` + `for line in proc.stdout` |
| Async | `WaitForExitAsync()` | `asyncio.create_subprocess_exec` |

## Exercises

1. Write `run_python(code: str, *args: str, timeout: float = 5) -> str` that runs `code` in a child Python with the given arguments, raises on non-zero exit or timeout, and returns stripped stdout. Test it with `"import sys; print(len(sys.argv) - 1)"` and the arguments `"a b"` and `"c"` (expect `"2"`).
2. Call `run_python("import sys; sys.exit(4)")` and catch the error, printing the return code. Then call it with code that sleeps for 2 seconds and `timeout=0.2`, and catch that error too.
3. Use `Popen` to run a child that prints the numbers 1–5 (flushing each line) and stop reading — and terminate the child with `proc.terminate()` — as soon as you have seen the number 3. Print the lines you received.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import subprocess
import sys

# 1 — list args, check=True, timeout
def run_python(code: str, *args: str, timeout: float = 5) -> str:
    result = subprocess.run([sys.executable, "-c", code, *args],
                            capture_output=True, text=True, check=True, timeout=timeout)
    return result.stdout.strip()

print(run_python("import sys; print(len(sys.argv) - 1)", "a b", "c"))   # 2

# 2 — two different exceptions
try:
    run_python("import sys; sys.exit(4)")
except subprocess.CalledProcessError as e:
    print("failed with", e.returncode)                  # 4
try:
    run_python("import time; time.sleep(2)", timeout=0.2)
except subprocess.TimeoutExpired as e:
    print(f"timed out after {e.timeout}s")

# 3 — stop early and terminate the child
counter = "import time\nfor i in range(1, 6):\n    print(i, flush=True)\n    time.sleep(0.05)"
seen = []
with subprocess.Popen([sys.executable, "-c", counter], stdout=subprocess.PIPE, text=True) as proc:
    for line in proc.stdout:
        seen.append(line.strip())
        if line.strip() == "3":
            proc.terminate()
            break
print(seen)                                             # ['1', '2', '3']
```

</details>